## Column Generation example

In [1]:
using HiGHS
using JuMP
using Graphs
using GraphPlot
using Plots

In [2]:
mutable struct Label

    C :: Float64
    R :: Vector{Float64}
    s :: Integer
    V :: Vector{Float64}

    Label(C, R, V) = new(C, copy(R), sum(0 .< V), copy(V))

end

In [3]:
Label(L, n) = Label(Inf, zeros(Int, L), zeros(Int, n))

Label

In [4]:
import Base

In [5]:
"""
Essa funcao implementa os rotulos dominados, ou seja, `x ≤ y` se e somente se `y` for dominado por `x`
"""

function Base.:<=(x :: Label, y :: Label) # essa funcao verifica quem do label e' <= (verifica qual rotulo e dominado pelo outro)

    return (x.C <= y.C) && all(x.R .<= y.R) && (x.s <= y.s) && all(p -> (p[1] == 0) || (p[2] > 0), zip(x.V, y.V))

end

In [6]:
function Base.isequal(x :: Label, y :: Label) # essa funcao verifica que o label e' == (verifica a igualdade entre os dois rotulos)

    # Atencao ao uso de == para numeros reais!!
    return (x.C == y.C) && all(x.R .== y.R) && (x.s == y.s) && all(x.V .== y.V)

end

In [7]:
function Base.copy(λ :: Label) # recebe um objeto λ, onde cria um novo objeto com os mesmos valores de C, R, V e retorna uma nova instancia

    return Label(λ.C, λ.R, λ.V)

end

In [8]:
# funcoes auxiliares
fromvec(v) = Label(v[1], [v[3]], v[4:end])
tovec(λ) = [λ.C; λ.s; λ.R...; λ.V...]

tovec (generic function with 1 method)

## ESPPRC

In [9]:
"""
Funcionalidade: aplica o algoritmo de correção de rótulos em um gráfico totalmente conectado para o ESPPRC.


### Entrada:
V - conjunto de vértices do gráfico;
S - matriz nxn onde cada entrada representa o valor objetivo obtido ao viajar ao longo de cada aresta do gráfico;
T - matriz nxn onde T[i,j] corresponde ao tempo usado para viajar do destino de i para a origem de j + tempo da
origem de j para o destino de j;
L - Float64 correspondendo ao limite superior para o recurso de tempo;
W - matriz nx2 contendo a janela de tempo de cada vértice;
Lambda - matriz nx1 onde cada entrada é o conjunto de rótulos do vértice correspondente;
n - número de vértices do gráfico.

### Saida:
   Lambda - matriz nx1 onde cada entrada é o conjunto de rótulos do vértice correspondente;
    bests - valor da função objetivo ótima (o rótulo inteiro)

"""
function ESPPRC(V, S, T, L, W, Λ, n)
    
    bests = Label(1, n)
    
    i = 1
    
    j = 1
    
    E = zeros(n)

    E[1] = 1

    F = Set([])
    
    changed = false
    
    while sum(E) != 0
        
        if E[i] != 0
                
            for k in 0:(n-2) # i é o vértice que estamos tratando e k varia ao longo de 0 a n-2
                
                j = (i+k) % n + 1  # j é um vértice ao qual podemos estender uma rótula
                                
                for λ in Λ[i]

                    if λ.V[j] == 0 # se a extensão for possível, fazemos isso
                        
                        F = union(F, Set([Extend(λ, i, j, S, T, L, W)])) 

                    end
                    
                end
                
                # println("Vertex $j")
                Λ[j], changed, bests = EFF(Λ[j], F, bests)

                F = Set([])

                if changed
                    
                    E[j] = 1
                    
                end
                
            end
            
            E[i] = 0
            
        end
        
        i = i % n + 1
        
    end

    return Λ, bests

end

ESPPRC

## Extend

In [10]:
"""
Funcionalidade: estender um rótulo do vértice i ao vértice j (por certificação em ESPPRC(), a extensão sempre será possível)

### Entrada:
lambda_i - rótulo da matriz (2+n)x1 a ser estendido de i;
i - vértice que contém o rótulo atual;
j - vértice que receberá um novo rótulo;
T - matriz nxn onde T[i,j] corresponde ao tempo usado para viajar do destino de i para a origem de j + tempo da
origem de j para o destino de j;
L - Float64 correspondendo ao limite superior para o recurso de tempo;
W - matriz nx2 contendo a janela de tempo de cada vértice.

### Saída:
lambda_j - matriz (2+n)x1 (rótulo estendido).
"""
function Extend(λ_i, i, j, S, T, L, W)
    
    viavel = true

    λ_j = copy(λ_i)

    λ_j.R[1] = max(W[j, 1], λ_j.R[1] + T[i,j]) # atualizamos o tempo percorrido e marcamos j como inacessível se lambda_j[2] + T[i,j] < W[j, 1] significa

    # que chegamos na tarefa antes do início da janela de tempo de execução da tarefa, ou seja, devemos esperar
    
    # atualizando a ordem em que a tarefa j foi concluída
    
    order = 1
    
    for k in λ_j.V
        if (k != Inf) && (k > order)
            order = k
        end
    end
    
    λ_j.V[j] = order + 1

    # atualizando o uso de recursos
    
    # a primeira entrada corresponde ao valor da função objetivo (número de tarefas concluídas)

    λ_j.C += S[i,j]
    
    # atualizando vértices inalcançáveis ​​de j agora  
    
    for k in 1:n
        
        if λ_j.V[k] == 0

            # Levamos em consideração o tempo que um veículo teria que esperar para concluir duas tarefas consecutivas
            
            if λ_j.R[1] + T[j,k] > W[k,2] || λ_j.R[1] + T[j,k] < W[k,1] - 5 || (λ_j.R[1] + T[j,k]) > L # Aqui, 5 é o limite imposto para o tempo de marcha lenta do veículo

                λ_j.V[k] = Inf
                
            end

        end
        
    end
    
    λ_j.s = sum(λ_j.V .> 0) # medindo vértices inacessíveis (visitados, impossíveis de visitar ou não tão bons para visitar)  
        
    return λ_j
    
end

Extend

## EFF

In [11]:
"""
Funcionalidade: mantém os conjuntos Lambda totalmente incomparáveis. Verifica qual rótulo pode entrar e aplicar a regra de dominação para remover
rótulos do conjunto.

### Entrada:
Lambda - conjunto de rótulos de um vértice;
F - conjunto de rótulos recém-estendidos (que possivelmente entrarão no Lambda);
bests - melhor valor de função objetivo (o rótulo inteiro) obtido até agora.

### Saída:
tmplambda - define Lambda possivelmente alterado;
changed - booleano que armazena se Lambda foi alterado;
bests - melhor valor de função objetivo (o rótulo inteiro) obtido até agora.
"""
function EFF(Λ :: Set, F :: Set, bests)
        
    dominated = false
    
    changed = false
    
    tmplambda = copy(Λ)
    
    if isempty(Λ) && !isempty(F)
        
        changed = true 
        
        for λ in F
            if λ.C < bests.C
                bests = λ
            end
        end
        
        return F, changed, bests
        
    else
        
        if isempty(F)
            
            changed = false
            
            return Λ, changed, bests
            
        else
            
            for λ_f in F
                
                for λ in tmplambda 

                    if λ <= λ_f && λ != λ_f
                        
                        dominated = true
                        
                        println("Dominated $(λ_f) by $(λ)")
                        
                    end
                    
                    
                    if (λ == λ_f) || dominated
                        
                        break
                        
                    end
                    
                    
                end
                
                if !dominated && !(λ_f in tmplambda)
                    
                    changed = true
                                        
                    for λ in tmplambda

                        if λ_f <= λ
                            
                            dominated = true 
                            
                            println("Dominated $(λ) by $(λ_f)")
                            
                            if dominated
                            
                                tmplambda = setdiff(tmplambda, Set([λ]))
                                
                            end
                            
                        end
                        
                    end
                    
                    tmplambda = union!(tmplambda, Set([λ_f]))
                                        
                    if λ_f.C < bests.C
                        
                        bests = λ_f
                    
                    end
                    
                end
                
                dominated = false
            
            end
            
        end
        
    end
    
    return tmplambda, changed, bests
    
end

EFF

## SIMPLEX

In [12]:
function simplexrev(A,b,c,base)

    k=0 # contador de iterações

    (m,n)=size(A) # dimensões da matriz do problema
    solucao=vec(zeros(n)) # criando um vetor com zeros para receber a solução

    while k<30

        B=A[:,base] # obtendo a matriz base

        cb=c[base] # vetor custo das variáveis básicas

        xb=B\b # atualizando xb

        y=B' \ cb # atualizando y, B'y = cb

        # encontrando o vetor com as colunas das variáveis não-básicas
        vnb=vec([1:n;]) # criando um vetor com o número de colunas de 1 a n
        for i=1:length(base) # os correspondentes da base recebem valor 0
           vnb[base[i]]=0
        end
        vnb=findall(x -> x>0,vnb) # os que não são zero (são positivos) formam vnb
        cnb=c[vnb] # cnb é a parte de c correspondente as variáveis não básicas

        D=A[:,vnb] # matriz com as colunas não básicas

        rn =  D'*y - cnb # rn = (zn - cn)' # custo relativo das variáveis não básicas

        

        # verificando a otimalidade
        if maximum(rn)<=0
            solucao[base]=xb
            votimo=c'*solucao   # ou cb'*xb

            # Cálculo do custo reduzido final
            zj = (c[base]' * (B \ A))'  # Zj = CB * B⁻¹ * Aj
            custo_reduzido = zj - c  # Zj - Cj
            
            println("Solução ótima=",solucao)
            println("Valor ótimo=",votimo)
            println("Iterações=",k)
            println("Base=",base)
            println("Zj - Cj (Custo reduzido final) = ", custo_reduzido) # Exibir custo reduzido na solução ótima
            return solucao, custo_reduzido
            
        end

        # encontrar quem entra na base
        aux=findmax(rn); # vnb(aux) que entrará na base
        ientra=vnb[aux[2]];
        colentra=A[:,ientra]; # selecionando a coluna a entrar na base A[:,ientra] ou D[:,aux]

        # y = inv(B)*colentra
        yentra = B\colentra # resolvendo o sistema B*yentra = colentra
        auxentra=findall(x -> x>0,yentra) # obtendo os indices do vetor y que são positivos

        if length(auxentra)==0  # se nenhum y_i é positivo o problema é ilimitado
            println("Problema Ilimitado")
            return
        end

        divi=xb[auxentra]./yentra[auxentra]  # Cálculo do quociente xb(i)/y(i) para y(i)>0
        aux1=findmin(divi) # o índice no qual o mínimo ocorre,aux1[2], é que indicará quem sai da base

        isai=base[auxentra[aux1[2]]] #base[auxentra[aux1[2]]] deve sair da base
        posisai=findall(x -> x==isai,base)
        base[posisai[1]]=ientra # atualizando os índices das colunas que formam a base

        k=k+1 # atualizando o contador
    end

end

simplexrev (generic function with 1 method)

## Data creation

In [13]:
# Criando dados para a pequena instância

# matriz de distância C (entre locais de entrega)

m = 4 # número de sites

r = 5 # número de tarefas originais (não contendo tarefas artificiais)

n = r + 2 # numero de tarefas considerando o ponto de saida e de chegada (tarefas artificiais)

C = Array{Float64}(undef, m, m) # a matriz C representa a distancia entre as cidades

for i in 1:m

    C[i,i] = 0.0

end

C[1,2] = 2 

C[1,3] = 6

C[1,4] = 7

C[2,3] = 4

C[2,4] = 9

C[3,4] = 5

for i in 0:m-1

    for j in 0:m-1

        C[m-i,m-j] = C[m-j,m-i]

    end

end

task = Array{Int64}(undef, r, 2) # o que cada tarefa faz 

task[1,:] = [1, 2]

task[2,:] = [2, 3]

task[3,:] = [3, 4]

task[4,:] = [4, 1]

task[5,:] = [1, 3]

L = 18.0 # correspondendo ao limite superior para o recurso de tempo

W = Array{Float64}(undef, n, 2) # matriz que representa o tempo da tarefa

W[1,:] = [0, L]

W[2,:] = [2.0, 5.0]

W[3,:] = [0, 4.0]

W[4,:] = [0, 9.0]

W[5,:] = [0, 18.0]

W[6, :] = [0,11.0]

W[7, :] = [0, L]



2-element Vector{Float64}:
  0.0
 18.0

In [14]:
"""
Funcionalidade: cria dados de acordo com parâmetros para o ESPPRC.

### Entrada:
n - número de tarefas(certificados) do gráfico (incluindo origem e destino);
L - limite superior do Float64 para o recurso de tempo;
dual - matriz 4x1 incluindo variáveis ​​duais (em ordem: lambda_0, lambda_1, lambda_2 E lambda_3)
C - matriz mxm correspondente à matriz de distância entre as m cidades;
task - matriz(n-2)x2 onde a primeira coluna corresponde à origem e a segunda coluna ao destino da tarefa representada pela i-ésima linha
W - matriz nx2 com intervalo de execução de cada tarefa

### Saída:
V, S, T, L, W, Lambda, n - saída que será a entrada correspondente do ESPPRC (verifique a documentação do ESPPRC)
"""
function data(n, L, dual, C, task, W)

    V = Set([i for i in 1:n])

    
    T = zeros(n,n)

    for i in 1:n

        T[i,i] = 100

    end

    T[2:n-1,1] .= 100.0

    T[n,:] .= 100.0

    T[1:n-1,n] .= 0.0

    for i in 2:n-1

        for j in 2:n-1 

            if i!=j

                T[i,j] = C[task[i-1,2], task[j-1,1]] + C[task[j-1,1], task[j-1,2]]

            end

        end

    end


    for j in 2:n-1 

        # T[1,j] = C[task[1,1], task[j-1,1]] + C[task[j-1,1], task[j-1,2]]
        T[1,j] = C[task[j-1,1], task[j-1,2]]

    end


    Lambda = Array{Any,1}(undef, n)

    label_origin = Label(0, [0.0], zeros(n))

    label_origin.V[1] = 1.0

    Lambda[1] = Set([label_origin])

    for i in 2:n
        Lambda[i] = Set([])
    end

    S = Array{Float64}(undef, n, n)

    S[:, 1] .= 0

    S[:, end] .= 0

    S[1, end] = 0

    for i in 2:n

        for j in 2:n-1

            S[i,j] = -1 - dual[j]

        end

    end
    
    for j in 2:n-1

        S[1,j] = -1 - dual[j] - dual[1]

    end

    return V, S, T, L, W, Lambda, n
end

data

In [ ]:
# Usando C (a matriz que representa a distancia entre as cidades) e task (que representa o que cada tarefa faz)
m = size(C, 1)  # Número de vértices

# Criar um grafico direcionado
g = SimpleDiGraph(m)

# Criar a lista de arestas com base nas rotas válidas em 'task'
edge_list = []

for i in 1:size(task, 1)  # Percorre todas as tarefas
    origem, destino = task[i, 1], task[i, 2]
    
    if C[origem, destino] > 0  # Só adiciona se existir uma distância válida em C
        push!(edge_list, (origem, destino, C[origem, destino]))
    end
end

# Adicionar arestas no grafico
weights = Dict()
for (u, v, w) in edge_list
    add_edge!(g, u, v)
    add_edge!(g, v, u)  # Adiciona a aresta reversa (pra nao precisar escrever o a rota inversa)
    weights[(u, v)] = w # (weights) guarda os pesos que nesse caso representa o tempo entre uma cidade e outra 
    weights[(v, u)] = w
end

# Obter as arestas e extrair os pesos
graph_edges = collect(Graphs.edges(g))
edge_labels = [weights[(u.src, u.dst)] for u in graph_edges]

# Plotar o grafico
gplot(g, nodelabel=1:nv(g), edgelabel=edge_labels)



In [ ]:
A=[1 0 0 0 0 0 1 1; 0 1 0 0 0 0 0 1; 0 0 1 0 0 0 1 0; 0 0 0 1 0 0 0 0; 0 0 0 0 1 0 0 0; 0 0 0 0 0 1 0 0] # precisa colocar manualmente

6×8 Matrix{Int64}:
 1  0  0  0  0  0  1  1
 0  1  0  0  0  0  0  1
 0  0  1  0  0  0  1  0
 0  0  0  1  0  0  0  0
 0  0  0  0  1  0  0  0
 0  0  0  0  0  1  0  0

In [ ]:
num_caminhoes = 2 # numero de caminhoes 
b = vcat(num_caminhoes, ones(size(A,1)-1))

6-element Vector{Float64}:
 2.0
 1.0
 1.0
 1.0
 1.0
 1.0

In [ ]:
base = collect(1:size(A,1)) # matriz da base 
#o comando collect transforma um intervalo em vetor,matriz

6-element Vector{Int64}:
 1
 2
 3
 4
 5
 6

In [ ]:
# Número de colunas e linhas
(m, n) = size(A)

# Encontrar índices das colunas fora da base
vnb = setdiff(collect(1:n), base)  # Colunas que não estão na base

# Criar c automaticamente
c = zeros(n)  # Inicializa um vetor de zeros com tamanho n
for j in vnb
    c[j] = -sum(A[2:end, j])  # Soma dos elementos da coluna sem a primeira linha, com sinal negativo
end

println(c)

[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -1.0, -1.0]


In [ ]:
base_inicial = copy(base)

6-element Vector{Int64}:
 1
 2
 3
 4
 5
 6

In [ ]:
solucao, custo_reduzido = simplexrev(A,b,c,base)

Solução ótima=[0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0]
Valor ótimo=-2.0
Iterações=2
Base=[8, 2, 7, 4, 5, 6]
Zj - Cj (Custo reduzido final) = [-1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


([0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0], [-1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])

In [ ]:
base_inicial

6-element Vector{Int64}:
 1
 2
 3
 4
 5
 6

## Examples

In [ ]:
V, S, T, L, W, Lambda, n = data(7, 18, custo_reduzido[base_inicial], C, task, W)
display(T)
rotulos, melhor = ESPPRC(V, S, T, L, W, Lambda, n)

7×7 Matrix{Float64}:
 100.0    2.0    4.0    5.0    7.0    6.0    0.0
 100.0  100.0    4.0    9.0   16.0    8.0    0.0
 100.0    8.0  100.0    5.0   12.0   12.0    0.0
 100.0    9.0   13.0  100.0    7.0   13.0    0.0
 100.0    2.0    6.0   11.0  100.0    6.0    0.0
 100.0    8.0    8.0    5.0   12.0  100.0    0.0
 100.0  100.0  100.0  100.0  100.0  100.0  100.0

Dominated Label(0.0, [2.0], 7, [1.0, 2.0, Inf, Inf, Inf, Inf, 3.0]) by Label(0.0, [0.0], 7, [1.0, Inf, Inf, Inf, Inf, Inf, 2.0])
Dominated Label(-1.0, [18.0], 6, [1.0, 2.0, Inf, Inf, 3.0, Inf, 0.0]) by Label(-1.0, [16.0], 6, [1.0, Inf, 2.0, Inf, 3.0, Inf, 0.0])
Dominated Label(0.0, [4.0], 7, [1.0, Inf, 2.0, Inf, Inf, Inf, 3.0]) by Label(0.0, [0.0], 7, [1.0, Inf, Inf, Inf, Inf, Inf, 2.0])
Dominated Label(-1.0, [16.0], 6, [1.0, Inf, 2.0, Inf, 3.0, Inf, 0.0]) by Label(-2.0, [16.0], 6, [1.0, Inf, 2.0, 3.0, 4.0, Inf, 0.0])
Dominated Label(0.0, [5.0], 7, [1.0, Inf, Inf, 2.0, Inf, Inf, 3.0]) by Label(0.0, [0.0], 7, [1.0, Inf, Inf, Inf, Inf, Inf, 2.0])
Dominated Label(-1.0, [12.0], 7, [1.0, Inf, Inf, 2.0, 3.0, Inf, 4.0]) by Label(-1.0, [9.0], 7, [1.0, Inf, 2.0, 3.0, Inf, Inf, 4.0])
Dominated Label(0.0, [7.0], 7, [1.0, Inf, Inf, Inf, 2.0, Inf, 3.0]) by Label(0.0, [0.0], 7, [1.0, Inf, Inf, Inf, Inf, Inf, 2.0])
Dominated Label(0.0, [6.0], 7, [1.0, Inf, Inf, Inf, Inf, 2.0, 3.0]) by Label(0.0, [0.0

(Any[Set(Label[Label(0.0, [0.0], 0, [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])]), Set(Any[Label(0.0, [2.0], 4, [1.0, 2.0, Inf, Inf, 0.0, 0.0, 0.0])]), Set(Any[Label(0.0, [4.0], 4, [1.0, Inf, 2.0, 0.0, 0.0, Inf, 0.0])]), Set(Any[Label(-1.0, [9.0], 5, [1.0, Inf, 2.0, 3.0, 0.0, Inf, 0.0]), Label(0.0, [5.0], 5, [1.0, Inf, Inf, 2.0, 0.0, Inf, 0.0])]), Set(Any[Label(-2.0, [16.0], 6, [1.0, Inf, 2.0, 3.0, 4.0, Inf, 0.0]), Label(-1.0, [12.0], 6, [1.0, Inf, Inf, 2.0, 3.0, Inf, 0.0]), Label(0.0, [7.0], 6, [1.0, Inf, Inf, Inf, 2.0, Inf, 0.0])]), Set(Any[Label(0.0, [6.0], 5, [1.0, Inf, Inf, Inf, 0.0, 2.0, 0.0]), Label(-1.0, [10.0], 6, [1.0, 2.0, Inf, Inf, Inf, 3.0, 0.0])]), Set(Any[Label(-1.0, [9.0], 7, [1.0, Inf, 2.0, 3.0, Inf, Inf, 4.0]), Label(0.0, [0.0], 7, [1.0, Inf, Inf, Inf, Inf, Inf, 2.0]), Label(-2.0, [16.0], 7, [1.0, Inf, 2.0, 3.0, 4.0, Inf, 5.0])])], Label(-2.0, [16.0], 6, [1.0, Inf, 2.0, 3.0, 4.0, Inf, 0.0]))

In [ ]:
for lambda in rotulos[end]
    if lambda.C < 0
        println(lambda.V)
    end 
end

[1.0, Inf, 2.0, 3.0, Inf, Inf, 4.0]
[1.0, Inf, 2.0, 3.0, 4.0, Inf, 5.0]


In [ ]:
#vamos pegar o lambda.V que lambda.C < 0 e transformar em uma matriz coluna 
valores_coluna = collect([lambda.V[1:end-1] for lambda in rotulos[end] if lambda.C < 0])

valores_coluna = [map(x -> isinf(x) ? 0.0 : 1.0, v) for v in valores_coluna]   
#o comando isinf(x) verifica se o numero e' infinito, ai nesse caso se for inf ele faz a mudanca pra zero pra todo x que pertencer ao vetor 
#o comando map faz com que seja aplicada uma funcao a cada elemento do vetor 

valores_coluna = reshape(valores_coluna, :, 1)  # Garante que seja uma coluna, onde o comando reshape faz com que a matriz seja de n linhas e apenas uma coluna 


2×1 Matrix{Vector{Float64}}:
 [1.0, 0.0, 1.0, 1.0, 0.0, 0.0]
 [1.0, 0.0, 1.0, 1.0, 1.0, 0.0]

In [ ]:
A = hcat(A, valores_coluna...)

6×10 Matrix{Float64}:
 1.0  0.0  0.0  0.0  0.0  0.0  1.0  1.0  1.0  1.0
 0.0  1.0  0.0  0.0  0.0  0.0  0.0  1.0  0.0  0.0
 0.0  0.0  1.0  0.0  0.0  0.0  1.0  0.0  1.0  1.0
 0.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0  1.0  1.0
 0.0  0.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0  1.0
 0.0  0.0  0.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0

SIMPLEX

In [ ]:
base.=base_inicial

6-element Vector{Int64}:
 1
 2
 3
 4
 5
 6

In [ ]:
# Número de colunas e linhas
(m, n) = size(A)

# Encontrar índices das colunas fora da base
vnb = setdiff(collect(1:n), base)  # Colunas que não estão na base

# Criar c automaticamente
c = zeros(n)  # Inicializa um vetor de zeros com tamanho n
for j in vnb
    c[j] = -sum(A[2:end, j])  # Soma dos elementos da coluna sem a primeira linha, com sinal negativo
end

println(c)

[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -1.0, -1.0, -2.0, -3.0]


In [ ]:
solucao, custo_reduzido = simplexrev(A,b,c,base)

Solução ótima=[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0]
Valor ótimo=-4.0
Iterações=2
Base=[8, 2, 10, 4, 5, 6]
Zj - Cj (Custo reduzido final) = [-1.0, 0.0, -2.0, 0.0, 0.0, 0.0, -2.0, 0.0, -1.0, 0.0]


([0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0], [-1.0, 0.0, -2.0, 0.0, 0.0, 0.0, -2.0, 0.0, -1.0, 0.0])

In [ ]:
V, S, T, L, W, Lambda, n = data(7, 18, custo_reduzido[base_inicial], C, task, W)
display(T)
rotulos, melhor = ESPPRC(V, S, T, L, W, Lambda, n)

7×7 Matrix{Float64}:
 100.0    2.0    4.0    5.0    7.0    6.0    0.0
 100.0  100.0    4.0    9.0   16.0    8.0    0.0
 100.0    8.0  100.0    5.0   12.0   12.0    0.0
 100.0    9.0   13.0  100.0    7.0   13.0    0.0
 100.0    2.0    6.0   11.0  100.0    6.0    0.0
 100.0    8.0    8.0    5.0   12.0  100.0    0.0
 100.0  100.0  100.0  100.0  100.0  100.0  100.0

Dominated Label(0.0, [2.0], 7, [1.0, 2.0, Inf, Inf, Inf, Inf, 3.0]) by Label(0.0, [0.0], 7, [1.0, Inf, Inf, Inf, Inf, Inf, 2.0])
Dominated Label(1.0, [9.0], 5, [1.0, Inf, 2.0, 3.0, 0.0, Inf, 0.0]) by Label(0.0, [5.0], 5, [1.0, Inf, Inf, 2.0, 0.0, Inf, 0.0])
Dominated Label(1.0, [16.0], 6, [1.0, Inf, 2.0, Inf, 3.0, Inf, 0.0]) by Label(0.0, [7.0], 6, [1.0, Inf, Inf, Inf, 2.0, Inf, 0.0])
Dominated Label(2.0, [4.0], 7, [1.0, Inf, 2.0, Inf, Inf, Inf, 3.0]) by Label(0.0, [0.0], 7, [1.0, Inf, Inf, Inf, Inf, Inf, 2.0])
Dominated Label(-1.0, [18.0], 6, [1.0, 2.0, Inf, Inf, 3.0, Inf, 0.0]) by Label(-1.0, [12.0], 6, [1.0, Inf, Inf, 2.0, 3.0, Inf, 0.0])
Dominated Label(0.0, [5.0], 7, [1.0, Inf, Inf, 2.0, Inf, Inf, 3.0]) by Label(0.0, [0.0], 7, [1.0, Inf, Inf, Inf, Inf, Inf, 2.0])
Dominated Label(0.0, [7.0], 7, [1.0, Inf, Inf, Inf, 2.0, Inf, 3.0]) by Label(0.0, [0.0], 7, [1.0, Inf, Inf, Inf, Inf, Inf, 2.0])
Dominated Label(-1.0, [12.0], 7, [1.0, Inf, Inf, 2.0, 3.0, Inf, 4.0]) by Label(-1.0, [10.0],

(Any[Set(Label[Label(0.0, [0.0], 0, [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])]), Set(Any[Label(0.0, [2.0], 4, [1.0, 2.0, Inf, Inf, 0.0, 0.0, 0.0])]), Set(Any[Label(2.0, [4.0], 4, [1.0, Inf, 2.0, 0.0, 0.0, Inf, 0.0])]), Set(Any[Label(0.0, [5.0], 5, [1.0, Inf, Inf, 2.0, 0.0, Inf, 0.0])]), Set(Any[Label(0.0, [7.0], 6, [1.0, Inf, Inf, Inf, 2.0, Inf, 0.0]), Label(-1.0, [12.0], 6, [1.0, Inf, Inf, 2.0, 3.0, Inf, 0.0])]), Set(Any[Label(-1.0, [10.0], 6, [1.0, 2.0, Inf, Inf, Inf, 3.0, 0.0]), Label(0.0, [6.0], 5, [1.0, Inf, Inf, Inf, 0.0, 2.0, 0.0])]), Set(Any[Label(-1.0, [10.0], 7, [1.0, 2.0, Inf, Inf, Inf, 3.0, 4.0]), Label(0.0, [0.0], 7, [1.0, Inf, Inf, Inf, Inf, Inf, 2.0])])], Label(-1.0, [18.0], 6, [1.0, 2.0, Inf, Inf, 3.0, Inf, 0.0]))

In [ ]:
for lambda in rotulos[end]
    if lambda.C < 0
        println(lambda.V)
    end
end

[1.0, 2.0, Inf, Inf, Inf, 3.0, 4.0]


In [ ]:
#vamos pegar o lambda.V que lambda.C < 0 e transformar em uma matriz coluna 
valores_coluna = collect([lambda.V[1:end-1] for lambda in rotulos[end] if lambda.C < 0])

valores_coluna = [map(x -> isinf(x) ? 0.0 : 1.0, v) for v in valores_coluna]   
#o comando isinf(x) verifica se o numero e' infinito, ai nesse caso se for inf ele faz a mudanca pra zero pra todo x que pertencer ao vetor 
#o comando map faz com que seja aplicada uma funcao a cada elemento do vetor 

valores_coluna = reshape(valores_coluna, :, 1)  # Garante que seja uma coluna, onde o comando reshape faz com que a matriz seja de n linhas e apenas uma coluna 


1×1 Matrix{Vector{Float64}}:
 [1.0, 1.0, 0.0, 0.0, 0.0, 1.0]

In [ ]:
A = hcat(A, valores_coluna...)

6×11 Matrix{Float64}:
 1.0  0.0  0.0  0.0  0.0  0.0  1.0  1.0  1.0  1.0  1.0
 0.0  1.0  0.0  0.0  0.0  0.0  0.0  1.0  0.0  0.0  1.0
 0.0  0.0  1.0  0.0  0.0  0.0  1.0  0.0  1.0  1.0  0.0
 0.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0  1.0  1.0  0.0
 0.0  0.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0  1.0  0.0
 0.0  0.0  0.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0  1.0

In [ ]:
base.=base_inicial

6-element Vector{Int64}:
 1
 2
 3
 4
 5
 6

In [ ]:
# Número de colunas e linhas
(m, n) = size(A)

# Encontrar índices das colunas fora da base
vnb = setdiff(collect(1:n), base)  # Colunas que não estão na base

# Criar c automaticamente
c = zeros(n)  # Inicializa um vetor de zeros com tamanho n
for j in vnb
    c[j] = -sum(A[2:end, j])  # Soma dos elementos da coluna sem a primeira linha, com sinal negativo
end

println(c)

[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -1.0, -1.0, -2.0, -3.0, -2.0]


In [ ]:
solucao, custo_reduzido = simplexrev(A,b,c,base)

Solução ótima=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0]
Valor ótimo=-5.0
Iterações=2
Base=[11, 2, 10, 4, 5, 6]
Zj - Cj (Custo reduzido final) = [-2.0, 0.0, -1.0, 0.0, 0.0, 0.0, -2.0, -1.0, -1.0, 0.0, 0.0]


([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0], [-2.0, 0.0, -1.0, 0.0, 0.0, 0.0, -2.0, -1.0, -1.0, 0.0, 0.0])

In [ ]:
V, S, T, L, W, Lambda, n = data(7, 18, custo_reduzido[base_inicial], C, task, W)
display(T)
rotulos, melhor = ESPPRC(V, S, T, L, W, Lambda, n)

7×7 Matrix{Float64}:
 100.0    2.0    4.0    5.0    7.0    6.0    0.0
 100.0  100.0    4.0    9.0   16.0    8.0    0.0
 100.0    8.0  100.0    5.0   12.0   12.0    0.0
 100.0    9.0   13.0  100.0    7.0   13.0    0.0
 100.0    2.0    6.0   11.0  100.0    6.0    0.0
 100.0    8.0    8.0    5.0   12.0  100.0    0.0
 100.0  100.0  100.0  100.0  100.0  100.0  100.0

Dominated Label(1.0, [2.0], 7, [1.0, 2.0, Inf, Inf, Inf, Inf, 3.0]) by Label(0.0, [0.0], 7, [1.0, Inf, Inf, Inf, Inf, Inf, 2.0])
Dominated Label(1.0, [9.0], 5, [1.0, Inf, 2.0, 3.0, 0.0, Inf, 0.0]) by Label(1.0, [5.0], 5, [1.0, Inf, Inf, 2.0, 0.0, Inf, 0.0])
Dominated Label(1.0, [16.0], 6, [1.0, Inf, 2.0, Inf, 3.0, Inf, 0.0]) by Label(1.0, [7.0], 6, [1.0, Inf, Inf, Inf, 2.0, Inf, 0.0])
Dominated Label(2.0, [4.0], 7, [1.0, Inf, 2.0, Inf, Inf, Inf, 3.0]) by Label(0.0, [0.0], 7, [1.0, Inf, Inf, Inf, Inf, Inf, 2.0])
Dominated Label(0.0, [18.0], 6, [1.0, 2.0, Inf, Inf, 3.0, Inf, 0.0]) by Label(0.0, [12.0], 6, [1.0, Inf, Inf, 2.0, 3.0, Inf, 0.0])
Dominated Label(1.0, [5.0], 7, [1.0, Inf, Inf, 2.0, Inf, Inf, 3.0]) by Label(0.0, [0.0], 7, [1.0, Inf, Inf, Inf, Inf, Inf, 2.0])
Dominated Label(1.0, [7.0], 7, [1.0, Inf, Inf, Inf, 2.0, Inf, 3.0]) by Label(0.0, [0.0], 7, [1.0, Inf, Inf, Inf, Inf, Inf, 2.0])
Dominated Label(0.0, [12.0], 7, [1.0, Inf, Inf, 2.0, 3.0, Inf, 4.0]) by Label(0.0, [0.0], 7, [

(Any[Set(Label[Label(0.0, [0.0], 0, [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])]), Set(Any[Label(1.0, [2.0], 4, [1.0, 2.0, Inf, Inf, 0.0, 0.0, 0.0])]), Set(Any[Label(2.0, [4.0], 4, [1.0, Inf, 2.0, 0.0, 0.0, Inf, 0.0])]), Set(Any[Label(1.0, [5.0], 5, [1.0, Inf, Inf, 2.0, 0.0, Inf, 0.0])]), Set(Any[Label(1.0, [7.0], 6, [1.0, Inf, Inf, Inf, 2.0, Inf, 0.0]), Label(0.0, [12.0], 6, [1.0, Inf, Inf, 2.0, 3.0, Inf, 0.0])]), Set(Any[Label(0.0, [10.0], 6, [1.0, 2.0, Inf, Inf, Inf, 3.0, 0.0]), Label(1.0, [6.0], 5, [1.0, Inf, Inf, Inf, 0.0, 2.0, 0.0])]), Set(Any[Label(0.0, [0.0], 7, [1.0, Inf, Inf, Inf, Inf, Inf, 2.0])])], Label(0.0, [0.0], 7, [1.0, Inf, Inf, Inf, Inf, Inf, 2.0]))

In [ ]:
for lambda in rotulos[end]
    if lambda.C < 0
        println(lambda.V)
    end
end